# Heap Memory
- Dynamic memory allocated at runtime (not compile-time)

- Lives in RAM, separate from stack memory

- Manually managed by the programmer (you're in control!)

# malloc()
- Stands for "memory allocation" - it's a function that requests memory from the heap

- Returns a pointer to the first byte of allocated memory

- Part of <stdlib.h> - always include this header!

- Allocates at runtime - size can be determined while program runs

In [3]:
%%bash
cat <<EOF > program.c
#include <stdio.h>
#include <stdlib.h>

int main(){
int a;    // stack variable - automatically managed
int *p;    // pointer on stack, will point to heap memory


// ===== FIRST ALLOCATION =====
    // malloc allocates memory on the heap
    // Returns void* pointer to the allocated memory
    // Optionally typecast to int* since we are storing integers not necessary
//p = (int*)malloc(sizeof(int)); 
p = malloc(1000); 
// Access heap memory through the pointer
*p = 10; // Store 10 in heap memory
printf("First value: %d\n", *p);  // Would print 10


// CRITICAL: Release heap memory back to OS
// Without this: MEMORY LEAK!
// The memory is freed, but p still holds the address


free(p); //if we dont do this older value of p would still sit in the memory


// p is now a DANGLING POINTER - points to freed memory!
// Do not do: *p = 20;  
// Would cause undefined behavior!
//p = NULL;  // Eliminates dangling danger



// ===== SECOND ALLOCATION =====
// Reuse the same pointer for new allocation
p=malloc(sizeof(int));
*p = 30;         // Store new value in new heap memory
printf("Second value: %d\n", *p);  // Would print 30

// ===== CLEANUP =====
free(p);         // Don't forget to free the second allocation too!
return 0;
}




EOF
gcc program.c -o program
./program

First value: 10
Second value: 30


In [10]:
=========== PROGRAM STACK + HEAP VISUALIZATION ===========

Step 1 — Start of main()

STACK
+-----------------------+
| main frame            |
| a (uninitialized)     |
| p (uninitialized)     |
+-----------------------+

HEAP
(empty)


Step 2 — First malloc

p = malloc(1000)

STACK
+-----------------------+
| main frame            |
| a (?)                 |
| p ----+               |
+-------|---------------+
        |
        v
HEAP
+-----------------------+
| 1000 bytes allocated  |
| heap block #1         |
+-----------------------+


Step 3 — Store value

*p = 10

STACK
+-----------------------+
| main frame            |
| a (?)                 |
| p ----+               |
+-------|---------------+
        |
        v
HEAP
+-----------------------+
| 10 | remaining bytes  |
+-----------------------+


Step 4 — free(p)

STACK
+-----------------------+
| main frame            |
| a (?)                 |
| p (dangling pointer)  |
+-----------------------+

HEAP
+-----------------------+
| freed memory block    |
+-----------------------+


Step 5 — Second malloc

p = malloc(sizeof(int))

STACK
+-----------------------+
| main frame            |
| a (?)                 |
| p ----+               |
+-------|---------------+
        |
        v
HEAP
+-----------------------+
| new int block         |
+-----------------------+


Step 6 — Store new value

*p = 30

STACK
+-----------------------+
| main frame            |
| a (?)                 |
| p ----+               |
+-------|---------------+
        |
        v
HEAP
+-----------------------+
| 30                    |
+-----------------------+


Step 7 — Final free(p)

STACK
+-----------------------+
| main frame            |
| a (?)                 |
| p (dangling pointer)  |
+-----------------------+

HEAP
(memory released)


SyntaxError: invalid syntax (2376731541.py, line 1)

# Now lets do the above example but not free the memory occupied by p during the 1st allocation

In [4]:
%%bash
cat <<EOF > program.c
#include <stdio.h>
#include <stdlib.h>

int main(){
int a;    // stack variable - automatically managed
int *p;    // pointer on stack, will point to heap memory


// ===== FIRST ALLOCATION =====
    // malloc allocates memory on the heap
    // Returns void* pointer to the allocated memory
    // Typecast to int* since we are storing integers
//p = (int*)malloc(sizeof(int)); 
p = malloc(1000); 
// Access heap memory through the pointer
*p = 10; // Store 10 in heap memory
printf("First value: %d\n", *p);  // Would print 10


// CRITICAL: Release heap memory back to OS
// Without this: MEMORY LEAK!
// The memory is freed, but p still holds the address


//free(p); //if we dont do this older value of p would still sit in the memory


// Now that we have lost the pointer to dynamically allocated memory above
// there is NO WAY to free it until our program ends!


// ===== SECOND ALLOCATION =====
// Reuse the same pointer for new allocation
// This would be new 4 byes of memory
// The previous 1000 bytes is still occupied in the memory but the pointer to
// those 1000 bytes is lost !!!
p=malloc(sizeof(int));
*p = 30;         // Store new value in new heap memory
printf("Second value: %d\n", *p);  // Would print 30

// ===== CLEANUP =====
free(p);         // Don't forget to free the second allocation too!
return 0;
}




EOF
gcc program.c -o program
./program

First value: 10
Second value: 30


In [11]:
=========== PROGRAM STACK + HEAP VISUALIZATION ===========

Step 1 — Start of main()

STACK
+-----------------------+
| main frame            |
| a (uninitialized)     |
| p (uninitialized)     |
+-----------------------+

HEAP
(empty)


Step 2 — First malloc

p = malloc(1000)

STACK
+-----------------------+
| main frame            |
| a (?)                 |
| p ----+               |
+-------|---------------+
        |
        v
HEAP
+-----------------------+
| 1000 bytes allocated  |  <- heap block #1
+-----------------------+


Step 3 — Store value

*p = 10

STACK
+-----------------------+
| main frame            |
| a (?)                 |
| p ----+               |
+-------|---------------+
        |
        v
HEAP
+-----------------------+
| 10 | remaining bytes  |
+-----------------------+


Step 4 — Lost pointer (no free)

p = malloc(sizeof(int))

STACK
+-----------------------+
| main frame            |
| a (?)                 |
| p ----+               |
+-------|---------------+
        |
        v
HEAP
+-----------------------+
| 1000 bytes (lost!)    | <- memory leak
+-----------------------+
| new int block          | <- heap block #2
+-----------------------+


Step 5 — Store new value

*p = 30

STACK
+-----------------------+
| main frame            |
| a (?)                 |
| p ----+               |
+-------|---------------+
        |
        v
HEAP
+-----------------------+
| 1000 bytes (lost!)    |
+-----------------------+
| 30                    |
+-----------------------+


Step 6 — Cleanup

free(p)

STACK
+-----------------------+
| main frame            |
| a (?)                 |
| p (dangling pointer)  |
+-----------------------+

HEAP
+-----------------------+
| 1000 bytes (lost!)    | <- memory leak persists
+-----------------------+
| freed memory block     |
+-----------------------+


End of program

STACK
(empty)

HEAP
+-----------------------+
| 1000 bytes (lost!)    | <- memory leak remains
+-----------------------+


SyntaxError: invalid syntax (698064321.py, line 1)

# calloc()

- Stands for "contiguous allocation" - allocates memory for an array of elements

- Initializes all bytes to zero - unlike malloc() which leaves garbage values

- Takes TWO arguments - number of elements and size of each element

- Part of <stdlib.h> - same family as malloc() and realloc()

In [8]:
#int *ptr = calloc(5, sizeof(int));
#                  ↑       ↑
#               number    size of
#               of items  each item

# Key Characteristics
- Zero-initialized - all bits set to 0 (useful for arrays, structures)

- Contiguous memory - elements stored sequentially

- Returns NULL on failure - always check!

- Slower than malloc() - has to write zeros to every byte

- Safer for certain uses - predictable initial state

- Built-in overflow protection - multiplication of arguments checked internally

In [29]:
%%bash
cat <<EOF > program.c

#include <stdio.h>
#include <stdlib.h>

int main() {
    int i;
    // Allocate memory for 5 integers and initialize to 0
    int *ptr = calloc(5, sizeof(int));

    if(ptr == NULL) {
        printf("Memory allocation failed!\n");
        return 1;
    }

    printf("Values after calloc (should all be 0): ");
    for(i = 0; i < 5; i++) {
        printf("%d ", ptr[i]);   // Print all 0s
    }
    printf("\n");

    // Assign some values
    ptr[0] = 10;
    ptr[1] = 20;
    ptr[2] = 30;

    printf("Values after assignment: ");
    for(i = 0; i < 5; i++) {
        printf("%d ", ptr[i]);
    }
    printf("\n");

    // Free memory
    free(ptr);

    return 0;
}
EOF

gcc program.c -o program
./program

Values after calloc (should all be 0): 0 0 0 0 0 
Values after assignment: 10 20 30 0 0 


# realloc()
- Stands for "re-allocation" - resizes previously allocated memory

- Can expand OR shrink existing memory blocks

- Preserves existing data when expanding

- Part of <stdlib.h> - same family as malloc()

# Basic Syntax


In [27]:
#ptr = realloc(ptr, new_size);
# ↑             ↑       ↑
# new pointer old ptr size in bytes

# Key Characteristics
- Preserves content - old data is copied to new location if needed

- May move memory - returns different address if can't expand in place

- Original pointer may become invalid - always use the returned pointer!

- Returns NULL on failure - but original memory is still allocated!



In [20]:
%%bash

cat <<EOF > program.c
#include<stdio.h>
#include<stdlib.h>

int main() {
    int i;
    int *ptr = (int *)malloc(2*sizeof(int));
    
    if(ptr == NULL)
    {
        printf("Memory not available!");
        exit(1);
    }
    
    // Using preset values instead of waiting for input
    printf("Setting first two numbers: 5 and 8 \n");
        *(ptr) = 5;
        *(ptr+1) = 8;
    
    //Memory allocation for 2 more integers
    ptr = (int *)realloc(ptr, 4*sizeof(int));
    if(ptr == NULL)
    {
        printf("Memory not available!");
        exit(1);
    }
    
    printf("Setting two more numbers: 11 and 15\n");
    *(ptr+2) = 11;
    *(ptr+3) = 15;
    
    
    //Printing the values on the screen
    printf("All four numbers: ");
    for(i=0; i<4; i++) {
        printf("%d ", *(ptr+i));
    }
    printf("\n");
    
    // Don't forget to free the memory!
    free(ptr);
    
    return 0;
}
EOF

gcc program.c -o program
./program

Setting first two numbers: 5 and 8 
Setting two more numbers: 11 and 15
All four numbers: 5 8 11 15 


In [24]:
=========== REALLOC STACK + HEAP VISUALIZATION ===========

Step 0 — Start of main()

STACK
+-------------------+
| main frame         |
| i (uninitialized)  |
| ptr (uninitialized)|
+-------------------+

HEAP
(empty)


Step 1 — First malloc (2 integers)

ptr = malloc(2*sizeof(int))

STACK
+-------------------+
| main frame         |
| i (?)              |
| ptr ----+          |
+--------|-----------+
         |
         v

HEAP
+-------------------+
| ? | ?             | <- 2 ints allocated
+-------------------+


Step 2 — Store first two numbers

*(ptr) = 5; *(ptr+1) = 8

STACK
+-------------------+
| main frame         |
| i (?)              |
| ptr ----+          |
+--------|-----------+
         |
         v

HEAP
+-------------------+
| 5 | 8             |
+-------------------+


Step 3 — realloc to 4 integers

ptr = realloc(ptr, 4*sizeof(int))

STACK
+-------------------+
| main frame         |
| i (?)              |
| ptr ----+          |
+--------|-----------+
         |
         v

HEAP
+-------------------+
| 5 | 8 | ? | ?     | <- 4 ints allocated
+-------------------+
(*old values preserved, new space uninitialized*)


Step 4 — Store two more numbers

*(ptr+2) = 11; *(ptr+3) = 15

STACK
+-------------------+
| main frame         |
| i (?)              |
| ptr ----+          |
+--------|-----------+
         |
         v

HEAP
+-------------------+
| 5 | 8 | 11 | 15   |
+-------------------+


Step 5 — Print all numbers

Output: 5 8 11 15

STACK / HEAP unchanged


Step 6 — Free memory

free(ptr)

STACK
+-------------------+
| main frame         |
| i (?)              |
| ptr (dangling)     |
+-------------------+

HEAP
(empty) <- memory returned to OS


End of program

SyntaxError: invalid syntax (3702708913.py, line 1)